# Shape optimization with exact adjoint gradients

This notebook walks through what `02_exact_shape_gradients.py` runs end to end, with room to
experiment. The claim being demonstrated: **shape derivatives of a real Kratos solve can be exact** -
not surrogate approximations, not finite differences - by differentiating through the solver's own
assembled FEM system.

Three steps:

1. **A sensitivity field.** For the objective $J = \sum_i T_i$ on a heat-conduction solve, the adjoint
   method gives $dJ/dX$ at *every node* for the cost of one linear solve.
2. **A chain rule to design variables.** A free-form deformation (FFD) lattice parameterizes the shape
   with 8 control points; `ComputeControlSensitivities` back-propagates the nodal field through the
   (differentiable, torch-based) deformation to $dJ/d(\mathrm{control})$ - verified below against
   re-solved finite differences.
3. **Optimization.** Twenty gradient-descent steps drive $J$ to 75 % of its initial value.

In [1]:
import contextlib
import os

import numpy
import torch
import matplotlib.pyplot as plt
import matplotlib.tri as mtri

import KratosMultiphysics as Kratos
Kratos.Logger.GetDefaultOutput().SetSeverity(Kratos.Logger.Severity.WARNING)
from KratosMultiphysics.PhysicsNeMoApplication import differentiable_residual
from KratosMultiphysics.PhysicsNeMoApplication import sensitivity_utils
from KratosMultiphysics.PhysicsNeMoApplication.mesh_bridge import deformation

import thermal_case

# FFD lattice spanning the unit square (z made non-degenerate on purpose)
ORIGIN = [0.0, 0.0, -0.5]
EXTENT = [1.0, 1.0, 1.0]


@contextlib.contextmanager
def Quiet():
    saved = os.dup(1)
    with open(os.devnull, "w") as null:
        os.dup2(null.fileno(), 1)
    try:
        yield
    finally:
        os.dup2(saved, 1)
        os.close(saved)

## The case and its objective

A stationary heat-conduction problem on the unit square ($-k\,\Delta u = f$, walls clamped to zero).
`Solve` optionally deforms the mesh through the FFD lattice first, so the *same* function serves the
verification (re-solve at perturbed controls) and the optimization loop.

In [2]:
def Solve(control=None, reference=None, divisions=12):
    model = Kratos.Model()
    with Quiet():
        analysis = thermal_case.CreateThermalAnalysis(
            model, conductivity=2.0, heat_flux=1.0, divisions=divisions)
        analysis.Initialize()
    model_part = model["ThermalModelPart"]

    if control is not None:
        points = torch.as_tensor(reference, dtype=torch.float64)
        deformed = deformation.DeformPoints(
            points, torch.as_tensor(control, dtype=torch.float64), "ffd",
            origin=ORIGIN, extent=EXTENT).numpy()
        for node, position in zip(model_part.Nodes, deformed):
            node.X0, node.Y0, node.Z0 = map(float, position)
            node.X, node.Y, node.Z = node.X0, node.Y0, node.Z0

    with Quiet():
        analysis.RunSolutionLoop()
    return model, model_part


def Objective(model_part):
    return sum(node.GetSolutionStepValue(Kratos.TEMPERATURE) for node in model_part.Nodes)


model, model_part = Solve()
reference = numpy.array([[node.X0, node.Y0, node.Z0] for node in model_part.Nodes])
J0 = Objective(model_part)
print(f"{model_part.NumberOfNodes()} nodes; J0 = {J0:.6f}")

169 nodes; J0 = 2.416546


## 1. The exact sensitivity field

`TangentAssembler` wraps the solver's own builder machinery; `ComputeShapeSensitivityField` runs the
adjoint: one solve of $K^T \lambda = \partial J / \partial u$, then element-local geometry
perturbations assemble $dJ/dX$ - the whole field at once, not one coordinate at a time.

In [3]:
def ShapeField(model_part):
    assembler = differentiable_residual.TangentAssembler(model_part)
    dof_map = differentiable_residual.DofFieldMap(
        assembler, [("TEMPERATURE", "node_historical")])
    dJ_du = numpy.ones(dof_map.n_equations)
    return sensitivity_utils.ComputeShapeSensitivityField(
        assembler, dof_map, dJ_du, fd_step=1e-6)


field = ShapeField(model_part)
points = numpy.array([[n.X, n.Y] for n in model_part.Nodes])
rows = {n.Id: i for i, n in enumerate(model_part.Nodes)}
triangles = numpy.array([[rows[e.GetGeometry()[i].Id] for i in range(3)]
                         for e in model_part.Elements])

fig, ax = plt.subplots(figsize=(5.4, 4.8))
ax.triplot(mtri.Triangulation(points[:, 0], points[:, 1], triangles),
           color="0.85", linewidth=0.4)
q = ax.quiver(points[:, 0], points[:, 1], field[:, 0], field[:, 1],
              numpy.hypot(field[:, 0], field[:, 1]), cmap="viridis", scale=4.0)
ax.set_aspect("equal"); ax.set_title("exact $dJ/dX$ at every node")
fig.colorbar(q, ax=ax, shrink=0.85)
plt.show()

/tmp/ipykernel_2153113/2182218303.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Chain rule to the FFD controls, verified

`ComputeControlSensitivities` is a vector-Jacobian product through the torch FFD: the nodal field
enters as the cotangent, and $dJ/d(\mathrm{control})$ comes out for all $2{\times}2{\times}2{\times}3$
lattice degrees of freedom at once. The check below re-solves the PDE at perturbed controls - the
most expensive and most convincing comparison there is.

In [4]:
control = numpy.zeros((2, 2, 2, 3))
gradient = sensitivity_utils.ComputeControlSensitivities(
    field, reference, control, "ffd", origin=ORIGIN, extent=EXTENT)

step = 1e-5
for entry in ((1, 0, 0, 0), (0, 1, 0, 1), (1, 1, 0, 0)):
    plus, minus = control.copy(), control.copy()
    plus[entry] += step
    minus[entry] -= step
    _, part_plus = Solve(plus, reference)
    _, part_minus = Solve(minus, reference)
    fd = (Objective(part_plus) - Objective(part_minus)) / (2 * step)
    rel = abs(float(gradient[entry]) - fd) / abs(fd)
    print(f"lattice{entry}: chain rule {float(gradient[entry]):+.8e}  "
          f"re-solve FD {fd:+.8e}  rel {rel:.2e}")
    assert rel < 1e-6

/home/vicente/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


lattice(1, 0, 0, 0): chain rule +6.00630469e-01  re-solve FD +6.00630468e-01  rel 6.97e-10
lattice(0, 1, 0, 1): chain rule +6.00630469e-01  re-solve FD +6.00630468e-01  rel 5.72e-10
lattice(1, 1, 0, 0): chain rule +5.79473079e-01  re-solve FD +5.79473079e-01  rel 3.98e-10


## 3. Twenty gradient steps

Minimize $(J - 0.75\,J_0)^2$ over the lattice (kept planar). Every iteration re-solves the PDE on the
deformed mesh and recomputes the exact gradient there - so this is true gradient descent on the
physics, with 8 active design variables and no surrogate in the loop.

In [5]:
TARGET = 0.75 * J0
LEARNING_RATE = 0.1

control = numpy.zeros((2, 2, 2, 3))
history = []
for iteration in range(20):
    _, part = Solve(control, reference) if iteration else (model, model_part)
    value = Objective(part)
    history.append(value)
    iteration_field = ShapeField(part)
    gradient = sensitivity_utils.ComputeControlSensitivities(
        iteration_field, reference, control, "ffd", origin=ORIGIN, extent=EXTENT)
    gradient[..., 2] = 0.0
    control = control - LEARNING_RATE * 2.0 * (value - TARGET) * gradient

_, final_part = Solve(control, reference)
history.append(Objective(final_part))
print(f"J: {history[0]:.6f} -> {history[-1]:.6f}   target {TARGET:.6f}")
assert abs(history[-1] - TARGET) < 5e-3 * J0, "the optimization must reach its target"

J: 2.416546 -> 1.812410   target 1.812410


In [6]:
final_points = numpy.array([[n.X, n.Y] for n in final_part.Nodes])
final_rows = {n.Id: i for i, n in enumerate(final_part.Nodes)}
final_triangles = numpy.array([[final_rows[e.GetGeometry()[i].Id] for i in range(3)]
                               for e in final_part.Elements])
final_T = numpy.array([n.GetSolutionStepValue(Kratos.TEMPERATURE) for n in final_part.Nodes])
initial_T = numpy.array([n.GetSolutionStepValue(Kratos.TEMPERATURE) for n in model_part.Nodes])

fig, axes = plt.subplots(1, 3, figsize=(13.2, 3.9))
axes[0].plot(history, "o-", ms=4)
axes[0].axhline(TARGET, color="r", ls="--", label="target")
axes[0].set_xlabel("iteration"); axes[0].set_ylabel("J"); axes[0].legend(); axes[0].grid(alpha=0.3)
for ax, (P, Tri, T, title) in zip(axes[1:], [
        (points, triangles, initial_T, "initial"),
        (final_points, final_triangles, final_T, "optimized")]):
    tr = mtri.Triangulation(P[:, 0], P[:, 1], Tri)
    m = ax.tricontourf(tr, T, levels=20, cmap="viridis", vmin=0.0, vmax=float(initial_T.max()))
    ax.triplot(tr, color="w", lw=0.3, alpha=0.5)
    ax.set_aspect("equal"); ax.set_title(title)
    fig.colorbar(m, ax=ax, shrink=0.85)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2153113/2579603474.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The optimizer pinches the domain: less area near the hot center means less integrated
temperature, exactly what the arrows of $dJ/dX$ pointed at. The final mesh stays valid - for larger
deformations the application's `mesh_bridge.deformation` module also provides mesh-quality energies
(including an inversion barrier) to keep an optimizer from tearing the mesh.